## Criação de Dataframe

In [1]:
# Versão do Spark Context
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("criando-dataframe") \
    .getOrCreate()

print(spark.version)

4.1.1


In [2]:
#Carregando dados em um Dataframe e Listando
from datetime import datetime, date
from pyspark.sql import Row

df = spark.createDataFrame([
    Row(Cidade='Salvador', Populacao = 6., Time = 'Bahia', Dat_cadastro=date(2021,10,10)),
    Row(Cidade='Recife', Populacao = 4., Time = 'Sport', Dat_cadastro=date(2021,10,11)),
    Row(Cidade='Fortaleza', Populacao = 3., Time = 'Ceara', Dat_cadastro=date(2021,10,12)),
    Row(Cidade='Natal', Populacao = 1., Time = 'ABC', Dat_cadastro=date(2021,10,13)),
    Row(Cidade='Joao Pessoa', Populacao = 2., Time = 'Botafogo', Dat_cadastro=date(2021,10,14))
])


In [ ]:
#Exibindo os dados do Dataframe

df.show()

+-----------+---------+--------+------------+
|     Cidade|Populacao|    Time|Dat_cadastro|
+-----------+---------+--------+------------+
|   Salvador|      6.0|   Bahia|  2021-10-10|
|     Recife|      4.0|   Sport|  2021-10-11|
|  Fortaleza|      3.0|   Ceara|  2021-10-12|
|      Natal|      1.0|     ABC|  2021-10-13|
|Joao Pessoa|      2.0|Botafogo|  2021-10-14|
+-----------+---------+--------+------------+



In [5]:
# Exibindo apenas dois registros no Dataframe
df.show(2)

+--------+---------+-----+------------+
|  Cidade|Populacao| Time|Dat_cadastro|
+--------+---------+-----+------------+
|Salvador|      6.0|Bahia|  2021-10-10|
|  Recife|      4.0|Sport|  2021-10-11|
+--------+---------+-----+------------+
only showing top 2 rows


In [6]:
# Exibindo tres registros do Dataframe na vertical
df.show(3, vertical=True)

-RECORD 0------------------
 Cidade       | Salvador   
 Populacao    | 6.0        
 Time         | Bahia      
 Dat_cadastro | 2021-10-10 
-RECORD 1------------------
 Cidade       | Recife     
 Populacao    | 4.0        
 Time         | Sport      
 Dat_cadastro | 2021-10-11 
-RECORD 2------------------
 Cidade       | Fortaleza  
 Populacao    | 3.0        
 Time         | Ceara      
 Dat_cadastro | 2021-10-12 
only showing top 3 rows


In [7]:
# Exibindo todos os dados com o collect()
df.collect()

[Row(Cidade='Salvador', Populacao=6.0, Time='Bahia', Dat_cadastro=datetime.date(2021, 10, 10)),
 Row(Cidade='Recife', Populacao=4.0, Time='Sport', Dat_cadastro=datetime.date(2021, 10, 11)),
 Row(Cidade='Fortaleza', Populacao=3.0, Time='Ceara', Dat_cadastro=datetime.date(2021, 10, 12)),
 Row(Cidade='Natal', Populacao=1.0, Time='ABC', Dat_cadastro=datetime.date(2021, 10, 13)),
 Row(Cidade='Joao Pessoa', Populacao=2.0, Time='Botafogo', Dat_cadastro=datetime.date(2021, 10, 14))]

In [8]:
# Sumarizando os dados a partir de determinados campos
df.select("Cidade", "Populacao").describe().show()

+-------+---------+------------------+
|summary|   Cidade|         Populacao|
+-------+---------+------------------+
|  count|        5|                 5|
|   mean|     NULL|               3.2|
| stddev|     NULL|1.9235384061671343|
|    min|Fortaleza|               1.0|
|    max| Salvador|               6.0|
+-------+---------+------------------+



In [9]:
# Exibindo o Schema do Dataframe
df.printSchema()

root
 |-- Cidade: string (nullable = true)
 |-- Populacao: double (nullable = true)
 |-- Time: string (nullable = true)
 |-- Dat_cadastro: date (nullable = true)



In [10]:
# Carregando o modulo de funções, expecificamente a função Upper
from pyspark.sql.functions import upper
df.withColumn('Cidade_Upper', upper(df.Cidade)).show()

+-----------+---------+--------+------------+------------+
|     Cidade|Populacao|    Time|Dat_cadastro|Cidade_Upper|
+-----------+---------+--------+------------+------------+
|   Salvador|      6.0|   Bahia|  2021-10-10|    SALVADOR|
|     Recife|      4.0|   Sport|  2021-10-11|      RECIFE|
|  Fortaleza|      3.0|   Ceara|  2021-10-12|   FORTALEZA|
|      Natal|      1.0|     ABC|  2021-10-13|       NATAL|
|Joao Pessoa|      2.0|Botafogo|  2021-10-14| JOAO PESSOA|
+-----------+---------+--------+------------+------------+



In [12]:
# Filtrando a Cidade de Salvador nos dados
df.filter(df.Cidade == 'Salvador').show()

+--------+---------+-----+------------+
|  Cidade|Populacao| Time|Dat_cadastro|
+--------+---------+-----+------------+
|Salvador|      6.0|Bahia|  2021-10-10|
+--------+---------+-----+------------+



In [14]:
# Criando uma tabela temporária em memória com os dados e utilizando consulta SQL
df.createOrReplaceTempView("Dados")
spark.sql("SELECT COUNT(*) AS Total FROM Dados").show()

+-----+
|Total|
+-----+
|    5|
+-----+



## Transformando um RDD em Dataframe

In [15]:
# carregando os dados sobre Capital de paises
pais = [("Brasil",10000),("Argentina",20000),("Australia",35000),("Italia",40000),("Egito",65000),("Mexico",80000)]
rddpais= spark.sparkContext.parallelize(pais)

In [16]:
# convertendo os dados do RDD para Dataframe com a operação toDF() 
dataframerdd= rddpais.toDF()

In [17]:
# Exibindo os dados do Dataframe
dataframerdd.show()

+---------+-----+
|       _1|   _2|
+---------+-----+
|   Brasil|10000|
|Argentina|20000|
|Australia|35000|
|   Italia|40000|
|    Egito|65000|
|   Mexico|80000|
+---------+-----+



In [18]:
# Criando o schema das colunas dos campos do Dataframe
Colunas = ["Pais","Total_capital(Bilhoes)"]
dataframerdd2= rddpais.toDF(Colunas)
dataframerdd2.printSchema()
dataframerdd2.show(truncate=False)

root
 |-- Pais: string (nullable = true)
 |-- Total_capital(Bilhoes): long (nullable = true)

+---------+----------------------+
|Pais     |Total_capital(Bilhoes)|
+---------+----------------------+
|Brasil   |10000                 |
|Argentina|20000                 |
|Australia|35000                 |
|Italia   |40000                 |
|Egito    |65000                 |
|Mexico   |80000                 |
+---------+----------------------+



In [1]:
# encerrando o Spark Session
spark.stop()

NameError: name 'spark' is not defined